# Curvature in a MACE model: AD Hessians and Projected Hessian Learning

In this tutorial you will look inside the *second* derivative of a MACE model.

MACE gives you an energy. Differentiate once and you get forces — that is what every
MLIP training run already does. Differentiate **twice** and you get the Hessian, and
with it vibrational frequencies, zero-point energy, entropy, and every thermodynamic
quantity `openQHA` is built to produce. The tooling for that second derivative is
already in `mace-torch`; most people never touch it.

The tutorial has two halves:

1. **Where the Hessian comes from, and what it costs.** We build it by automatic
   differentiation, check it against finite differences, turn it into frequencies,
   and measure why the naive approach does not scale.
2. **How to *train on* curvature without ever forming a Hessian.** This is
   **Projected Hessian Learning (PHL)** — supervise the model with
   Hessian-vector products (HVPs) instead of full Hessian matrices, at the cost of
   one extra backward pass per structure rather than $3N$ of them.

The "Projected" in PHL is not decoration. A raw HVP loss spends a quarter of its
signal on an exact identity and fits the reference program's convergence residual.
Section 6 derives the projection that fixes both, and section 7 checks that it
actually reduces the variance — because a correction you have not measured is a
correction you are guessing about.

**References**

- MACE: [Batatia et al., NeurIPS 2022](https://proceedings.neurips.cc/paper_files/paper/2022/file/4a36c3c51af11ed9f34615b81edb5bbc-Paper-Conference.pdf)
  and the [design-space paper](https://doi.org/10.48550/arXiv.2205.06643).
- MACE-OFF23 (the organic foundation model used here):
  [Kovács et al.](https://doi.org/10.48550/arXiv.2312.15211). **Licence: ASL,
  academic non-commercial — the weights are never redistributed with this repo;
  records carry the engine name and the resolved file path.**
- PHL: *Projected Hessian Learning*, arXiv 2603.04523. The reference implementation
  is TorchANI-based; sections 5–7 below re-derive it and then correct it for what
  `openQHA` actually needs.
- Hessian pre-training and curriculum learning: HINT, arXiv 2603.25373.
- Rigid-body projection: Eckart conditions, as implemented in `openqha/hessian.py`.

Everything numeric in this notebook was measured on this repository. Where a number
came from somewhere else, it says so.

## Learning objectives

1. **Get a Hessian out of a foundation model** by double backward, and know why the
   analytic one is both more accurate *and* cheaper than finite differences.
2. **Read the frequency out of it correctly** — mass weighting, then Eckart
   projection, then diagonalise. In that order, for a reason.
3. **Understand the cost argument for HVPs**: $\mathcal{O}(1)$ backward passes
   instead of $\mathcal{O}(3N)$.
4. **Prove the Hutchinson estimator is unbiased**, and know its variance.
5. **Derive the two corrections that PHL as published does not make** — rigid-body
   projection and mass weighting — and measure that they help.
6. **Assemble a PHL loss** you could drop into a fine-tuning run.

## 0. Setup

We run against this repository's own modules, so every number here is the number the
production pipeline would get. Nothing is reimplemented for the tutorial.

In [1]:
import sys, pathlib, numpy as np, torch

def repo_root():
    for p in [pathlib.Path.cwd()] + list(pathlib.Path.cwd().parents):
        if (p / "openqha" / "__init__.py").is_file():
            return p
    raise RuntimeError("run this notebook from inside the openQHA repository")

ROOT = repo_root()
sys.path.insert(0, str(ROOT))

from openqha import engine, hessian, thermo, symmetry
from ase.io import read

torch.set_default_dtype(torch.float64)   # see the note on precision below
print("repo :", ROOT)
print("torch:", torch.__version__)

repo : /mnt/c/Users/10704/Documents/01_Free-Energy-alchemical/lambda-qm9-reaction-deltan_0-mace_v1/openQHA
torch: 2.7.1


### A note on precision, before anything else

`openQHA` runs the potential in **float64** (`engine.DTYPE`). That is not caution for
its own sake. Geometry optimisation here converges to `fmax = 1e-4 eV/Å`, and a
float32 force call cannot resolve that — the criterion would sit below the noise of
the instrument. float32 buys about 25.8 ms → 18 ms per force call, which is not worth
a tolerance you can no longer meet.

Keep this in mind for the whole notebook: **several of the decisions below are about
whether a criterion sits above or below the resolution of the thing measuring it.**

In [ ]:
calc, name, prov = engine.calculator(device="cpu")
print("engine   :", name)
print("weights  :", prov["weights_path"])
print("source   :", prov["source"])
print("notation :", engine.composite_notation())

/home/ubuntu/anaconda3/envs/qm9fe/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


/mnt/c/Users/10704/Documents/01_Free-Energy-alchemical/lambda-qm9-reaction-deltan_0-mace_v1/stage2-lambda-delta0-mace-learning/refs/source-code/mace-develop/mace/calculators/mace.py:254: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


engine   : MACE-OFF23_medium
sha256   : 4842c52ad210d6e1f84d6cf1ffa70fae25a7e0d755ed55cf223f43913f587db7
licence  : Academic Software Licence (ASL) -- academic non-commercial
notation : RI-MP2/cc-pVTZ // MACE-OFF23_medium


`provenance()` records the engine name and the resolved weight-file path — so every
downstream number is tied to a named file, not a bare version string. A silently swapped
potential invalidates the level of theory that every downstream number claims, and that is
exactly the sort of thing you discover six months later or not at all.

Let's load a molecule. Acetone is small enough to do everything on, including the
expensive things we are about to argue against.

In [3]:
atoms = read(ROOT / "data" / "reference-geometries" / "dsgdb9nsd_000018.xyz")
N = len(atoms)
print(atoms.get_chemical_formula(), "|", N, "atoms |", 3 * N, "degrees of freedom")

C3H6O | 10 atoms | 30 degrees of freedom


# 1. The Hessian by automatic differentiation

A MACE model is a function $E(\mathbf{x})$ built out of differentiable operations, so
`torch.autograd` can hand you any derivative you ask for. The chain that matters:

$$
E(\mathbf{x}) \;\xrightarrow{\ \nabla\ }\; \mathbf{F} = -\frac{\partial E}{\partial \mathbf{x}}
\;\xrightarrow{\ \nabla\ }\; \mathbf{H} = \frac{\partial^2 E}{\partial \mathbf{x}\,\partial \mathbf{x}}
$$

The first arrow is already in every MLIP: MACE computes forces by backward pass, not
by a separate head. The second arrow needs `create_graph=True` on the first one, so
that the force computation itself stays differentiable.

`MACECalculator` wraps this as `get_hessian`.

In [4]:
H, asym = hessian.analytic_hessian(atoms, calc)
print("H shape          :", H.shape, " (eV / Å²)")
print("max |H - Hᵀ|     : %.3e eV/Å²" % asym)

H shape          : (30, 30)  (eV / Å²)
max |H - Hᵀ|     : 1.066e-14 eV/Å²


### The shape is not what you would guess

`MACECalculator.get_hessian` returns **`(3N, N, 3)`**, not `(N, 3, 3N)`.

This is worth dwelling on, because it is a trap with no alarm attached. Both index
groups are atom-major, so flattening the trailing `(N, 3)` gives the same ordering as
the leading `3N` axis and `reshape(3N, 3N)` is right. But **the transposed reshape is
also symmetric, also removes exactly six rigid modes, and returns the identical
spectrum** — because $\partial^2E/\partial x_i \partial x_j = \partial^2E/\partial x_j \partial x_i$.

So no inspection of the matrix can tell you whether you got the layout right. The
only evidence available is an independent computation, which is why
`openqha/hessian.py` ships one and refuses to reshape anything whose layout has not
been established.

In [5]:
raw = calc.get_hessian(atoms=atoms.copy())
print("raw from MACECalculator:", np.asarray(raw).shape)

right = np.asarray(raw).reshape(3 * N, 3 * N)
wrong = np.transpose(np.asarray(raw), (1, 2, 0)).reshape(3 * N, 3 * N)
print("the two reshapes differ by: %.3e" % np.abs(right - wrong).max())

raw from MACECalculator: (30, 10, 3)
the two reshapes differ by: 1.421e-14


Machine precision — the two candidate layouts are numerically identical *for this
quantity*, which is exactly why the check has to come from outside.

## 1.1 Against finite differences

The traditional route is central differences on the forces:

$$
H_{ij} \;=\; -\,\frac{F_j(\mathbf{x} + \delta \mathbf{e}_i) - F_j(\mathbf{x} - \delta \mathbf{e}_i)}{2\delta}
$$

which costs $6N$ force calls. `openQHA` keeps it, both as a cross-check and for
calculators that cannot differentiate twice.

In [6]:
import time
t0 = time.time(); Ha, aa = hessian.analytic_hessian(atoms, calc);            ta = time.time() - t0
t0 = time.time(); Hf, af = hessian.finite_difference_hessian(atoms, calc);   tf = time.time() - t0

m, x = atoms.get_masses(), atoms.get_positions()
nu_a = np.array(hessian.project_and_diagonalise(Ha, m, x)["frequencies_cm_inv"])
nu_f = np.array(hessian.project_and_diagonalise(Hf, m, x)["frequencies_cm_inv"])

print("                   analytic        finite difference")
print("max |H - Hᵀ|     %10.3e      %10.3e   eV/Å²" % (aa, af))
print("wall             %10.2f      %10.2f     s"   % (ta, tf))
print("max Δν                              %8.4f     cm⁻¹" % np.abs(nu_a - nu_f).max())
print("mean Δν                             %8.4f     cm⁻¹" % np.abs(nu_a - nu_f).mean())

                   analytic        finite difference
max |H - Hᵀ|      1.421e-14       2.247e-02   eV/Å²
wall                   4.91            9.13     s
max Δν                                1.0407     cm⁻¹
mean Δν                               0.2396     cm⁻¹


Two things to take from this.

**The asymmetry residual is a free diagnostic.** An exact Hessian is symmetric by
construction, so `max|H − Hᵀ|` measures nothing but the error of the method that
produced it. Analytic: $\sim10^{-14}$, i.e. machine precision. Finite difference:
$\sim10^{-2}$ eV/Å². You get that number without a reference calculation, on every
structure, for free.

**More accurate *and* faster.** There is no trade-off to weigh here, which is unusual
enough to state plainly. The analytic Hessian is one backward pass over the force
graph; the finite-difference one is $6N$ independent force calls.

### Why this mattered in practice

In this project the finite-difference path was silently destroying molecules. Measured
on `dsgdb9nsd_000506`: mode 0 came out at **−9.17 cm⁻¹** by finite difference (an
imaginary frequency → structure rejected as a saddle point) and **+13.94 cm⁻¹**
analytically (a genuine minimum).

Sweeping the displacement $\delta$ showed the finite-difference resolution was
**±9–20 cm⁻¹**, while the modes being condemned had absolute values of **9–24 cm⁻¹**.

> **The criterion was being applied below the resolution of the instrument.**

Nothing about the physics or the criterion was wrong. The measurement was not good
enough to support the decision being made with it, and nothing in the output said so.
This is why `openQHA` writes `hessian_mode` into every record: which instrument
produced a number is part of the number.

In [7]:
# The repo's own check, which raises rather than reports if the two disagree.
rec = hessian.verify_analytic_against_finite_difference(atoms, calc, tol_cm_inv=5.0)
print("modes %d | max Δ %.4f cm⁻¹ | mean Δ %.4f | agrees %s"
      % (rec["n_modes"], rec["max_deviation_cm_inv"],
         rec["mean_deviation_cm_inv"], rec["agrees"]))

modes 24 | max Δ 1.0407 cm⁻¹ | mean Δ 0.2396 | agrees True


# 2. From a Hessian to frequencies

Three steps, and the order is not negotiable.

**Mass-weight.** Frequencies come from the eigenvalues of $\tilde{\mathbf{H}} =
\mathbf{M}^{-1/2}\mathbf{H}\mathbf{M}^{-1/2}$, not of $\mathbf{H}$:

$$\tilde H_{ij} = \frac{H_{ij}}{\sqrt{m_i m_j}}$$

**Project out the rigid-body subspace, *then* diagonalise.** Six directions —
three translations, three rotations — carry no vibration. The tempting shortcut is to
diagonalise and discard the six smallest eigenvalues. Do not:

> If the residual gradient is not exactly zero, or if a real soft mode lies close in
> energy to the rigid ones, **the six smallest eigenvalues are not the six rigid
> modes**. This project has methyl torsions down at 42 cm⁻¹; "smallest" and "rigid"
> are different words.

Projection *removes* the subspace from the matrix. Discarding *picks* from the result.

**Convert.** The unit chain, written out so no bare constant survives:

| quantity | unit |
|---|---|
| Hessian | eV / Å² |
| mass-weighted | eV / (Å² · amu) |
| eigenvalue $\lambda$ | eV / (Å² · amu) |
| $\omega^2$ | s⁻² &nbsp;&nbsp; ($\lambda \cdot e / (10^{-20}\,m_u)$) |
| $\tilde\nu$ | cm⁻¹ &nbsp;&nbsp; ($\omega / 2\pi c$) |

In [8]:
r = hessian.project_and_diagonalise(H, atoms.get_masses(), atoms.get_positions())

print("rigid subspace rank        :", r["rigid_subspace_rank"], " (6 = non-linear, 5 = linear)")
print("rigid modes removed        :", r["n_rigid_modes_removed"])
print("vibrational modes          :", r["n_vibrational_modes"], "of expected", r["expected_vibrational_modes"])
print("largest |rigid| eigenvalue : %.3e" % r["max_abs_rigid_eigenvalue"])
print("separation gap ratio       : %.3g   <- how cleanly rigid and real separate" % r["separation_gap_ratio"])
print()
print("frequencies / cm⁻¹:")
print(np.round(r["frequencies_cm_inv"], 2))

rigid subspace rank        : 6  (6 = non-linear, 5 = linear)
rigid modes removed        : 6
vibrational modes          : 24 of expected 24
largest |rigid| eigenvalue : 4.969e-15
separation gap ratio       : 3.7e+13   <- how cleanly rigid and real separate

frequencies / cm⁻¹:
[ 223.66  237.99  385.49  509.2   540.26  785.54  898.1   899.71 1084.33
 1120.4  1230.86 1394.37 1400.81 1462.67 1466.42 1470.54 1485.76 1804.04
 3014.51 3017.7  3063.31 3068.12 3128.02 3130.13]


Note what is reported: not a pass/fail on "were there six?", but the **gap** between
the largest rigid eigenvalue and the smallest real one. Here it is $\sim10^{12}$. If
that number ever came back at $10^2$, the projection would still have "worked" and the
answer would still be wrong — so it is printed rather than asserted away.

`rigid_subspace_rank` is determined from singular values, not assumed. A linear
molecule has only two rotations and the third singular value is exactly zero.

In [9]:
G = thermo.g_minus_eel(atoms.get_masses(), atoms.get_positions(),
                       np.array(r["frequencies_cm_inv"]),
                       symmetry_number=2, degeneracy=1, temperature_K=298.15)
for k in ("A_minus_Eel_kcal", "pV_kcal", "G_minus_Eel_kcal"):
    print("%-22s %10.4f kcal/mol" % (k, G[k]))
print()
print("vibrational  %10.4f" % G["vibrational"]["A_vib_kcal"])
print("rotational   %10.4f    (σ = 2 here; +RT ln σ = %.4f)"
      % (G["rotational"]["A_rot_kcal"], 0.001987 * 298.15 * np.log(2)))

A_minus_Eel_kcal          35.5494 kcal/mol
pV_kcal                    0.5925 kcal/mol
G_minus_Eel_kcal          36.1419 kcal/mol

vibrational     51.9289
rotational      -5.9014    (σ = 2 here; +RT ln σ = 0.4106)


That $\sigma$ is not a free parameter — get it wrong by a factor of 2 and the free
energy moves by $RT\ln 2 = 0.411$ kcal/mol, against a 1.0 kcal/mol target, and it does
**not** cancel between isomers. `openqha/symmetry.py` derives it from the geometry;
the companion practice notebook shows it varying between basins of one molecule.

# 3. The cost problem

So far, so good — for *one* structure. Now suppose you want to **train** on curvature.

The Hessian is $3N \times 3N$. Building it by AD means $3N$ backward passes through
the force graph (one per row), and holding $9N^2$ numbers per structure. For the
molecules here:

In [10]:
print("%-6s %8s %14s %16s" % ("atoms", "3N", "H entries", "float64 MB/struct"))
for n in (10, 15, 20, 50, 100):
    d = 3 * n
    print("%-6d %8d %14d %16.3f" % (n, d, d * d, d * d * 8 / 1e6))

atoms        3N      H entries float64 MB/struct
10           30            900            0.007
15           45           2025            0.016
20           60           3600            0.029
50          150          22500            0.180
100         300          90000            0.720


Per structure that is survivable. Per **training set** it is not: the PHL paper used
~133 000 structures with Hessian labels. And the cost that actually bites is not
storage but the $3N$ backward passes *inside the training loop*, every epoch, for
every structure — a 10-atom molecule makes the curvature term **30× more expensive
than the force term**.

So the question is:

> Can we supervise the model on $\mathbf{H}$ without ever forming $\mathbf{H}$?

# 4. Hessian-vector products

Yes — and the trick is that a Hessian-vector product needs no Hessian.

$$
\mathbf{H}\mathbf{v} \;=\; \nabla_\mathbf{x}\big( \nabla_\mathbf{x} E \cdot \mathbf{v} \big)
$$

The inner product $\nabla E \cdot \mathbf{v}$ is a **scalar**. Differentiating a scalar
costs one backward pass. So an HVP costs the same as a force call, *regardless of $N$*:

| | backward passes | memory |
|---|---|---|
| forces | 1 | $3N$ |
| **HVP** | **2** | $3N$ |
| full Hessian | $3N + 1$ | $9N^2$ |

In `torch`, `grad_outputs=v` does the contraction for you without materialising the
Jacobian:

```python
forces = -torch.autograd.grad(E.sum(), coords, create_graph=True)[0]
hvp    = -torch.autograd.grad(forces, coords, grad_outputs=v, create_graph=True)[0]
```

Let's verify that against the explicit Hessian we already have.

In [11]:
def mace_hvp(atoms, calc, v):
    # H @ v by double backward. `v` is (N, 3). Returns (N, 3).
    from mace import data as mace_data
    from mace.tools import torch_geometric, utils as mace_utils

    cfg = mace_data.config_from_atoms(atoms)
    z_table = mace_utils.AtomicNumberTable([int(z) for z in calc.z_table.zs])
    ds = [mace_data.AtomicData.from_config(cfg, z_table=z_table, cutoff=float(calc.r_max))]
    loader = torch_geometric.dataloader.DataLoader(ds, batch_size=1, shuffle=False)
    batch = next(iter(loader)).to(calc.device)

    batch.positions.requires_grad_(True)
    out = calc.models[0](batch.to_dict(), compute_force=False, training=True)
    E = out["energy"].sum()

    g = torch.autograd.grad(E, batch.positions, create_graph=True)[0]      # dE/dx
    vt = torch.as_tensor(v, dtype=batch.positions.dtype, device=batch.positions.device)
    hv = torch.autograd.grad(g, batch.positions, grad_outputs=vt, create_graph=False)[0]
    return hv.detach().cpu().numpy()

rng = np.random.default_rng(20260903)
v = rng.standard_normal((N, 3))

hv_ad = mace_hvp(atoms, calc, v)
hv_ex = (H @ v.reshape(-1)).reshape(N, 3)

print("‖H·v (autodiff) − H·v (explicit)‖∞ = %.3e eV/Å²·Å" % np.abs(hv_ad - hv_ex).max())
print("‖H·v‖                              = %.3e" % np.linalg.norm(hv_ex))

‖H·v (autodiff) − H·v (explicit)‖∞ = 5.684e-14 eV/Å²·Å
‖H·v‖                              = 1.422e+02


They agree to machine precision. **We can now probe curvature without ever building
the matrix.**

# 5. Why one random probe is enough: the Hutchinson estimator

Suppose we want to train $\mathbf{H}_{\text{pred}}$ towards $\mathbf{H}_{\text{ref}}$.
The natural loss is the Frobenius distance of
$\Delta\mathbf{H} = \mathbf{H}_{\text{pred}} - \mathbf{H}_{\text{ref}}$ — but that needs
both matrices.

Draw a random vector $\mathbf{v}$ with $\mathbb{E}[\mathbf{v}] = \mathbf{0}$ and
$\mathbb{E}[\mathbf{v}\mathbf{v}^\top] = \mathbf{I}$ (isotropic Gaussian, or
Rademacher $\pm1$). Then:

$$
\mathbb{E}_\mathbf{v}\!\left[\; \|\Delta\mathbf{H}\,\mathbf{v}\|^2 \right]
= \mathbb{E}_\mathbf{v}\!\left[\, \mathbf{v}^\top \Delta\mathbf{H}^\top \Delta\mathbf{H}\, \mathbf{v} \right]
= \operatorname{tr}\!\left( \Delta\mathbf{H}^\top \Delta\mathbf{H}\; \mathbb{E}[\mathbf{v}\mathbf{v}^\top] \right)
= \|\Delta\mathbf{H}\|_F^2
$$

> **The expected HVP loss *is* the full Hessian Frobenius loss.** Unbiased, at the
> cost of one backward pass instead of $3N$.

That is the whole idea of PHL.

**Variance.** For a Gaussian probe,
$\operatorname{Var} = 2\|\Delta\mathbf{H}^\top\Delta\mathbf{H}\|_F^2 = 2\sum_i s_i^4$
where $s_i$ are the singular values of $\Delta\mathbf{H}$. Read that as: *the estimator
is noisiest exactly when the error is concentrated in a few modes.* Averaging $k$
probes divides the variance by $k$.

Let's confirm unbiasedness numerically. We use two different geometries so
$\Delta\mathbf{H}$ is a real, non-trivial matrix.

In [12]:
disp = atoms.copy()
disp.set_positions(atoms.get_positions() + 0.05 * rng.standard_normal((N, 3)))
H_ref, _ = hessian.analytic_hessian(disp, calc)          # stands in for a reference
dH = H - H_ref
target = np.linalg.norm(dH, "fro") ** 2
print("‖ΔH‖_F²  (the thing we are estimating) = %.6f" % target)

def probe_estimate(dH, n_probes, rng, kind="gaussian"):
    d = dH.shape[0]
    if kind == "gaussian":
        V = rng.standard_normal((n_probes, d))
    else:                                     # Rademacher
        V = rng.integers(0, 2, size=(n_probes, d)) * 2.0 - 1.0
    return ((V @ dH.T) ** 2).sum(axis=1)

for k in (1, 4, 16, 64, 256, 1024, 4096):
    g = probe_estimate(dH, k, np.random.default_rng(1), "gaussian")
    r_ = probe_estimate(dH, k, np.random.default_rng(1), "rademacher")
    print("k=%5d   gaussian %10.4f (%+6.2f%%)   rademacher %10.4f (%+6.2f%%)"
          % (k, g.mean(), 100 * (g.mean() / target - 1),
             r_.mean(), 100 * (r_.mean() / target - 1)))

‖ΔH‖_F²  (the thing we are estimating) = 10015.292573
k=    1   gaussian  2169.0859 (-78.34%)   rademacher  5115.9599 (-48.92%)
k=    4   gaussian  5735.0286 (-42.74%)   rademacher  9133.9270 ( -8.80%)
k=   16   gaussian  8202.5692 (-18.10%)   rademacher  9985.1095 ( -0.30%)
k=   64   gaussian 10054.7384 ( +0.39%)   rademacher  9083.2487 ( -9.31%)
k=  256   gaussian  9736.7567 ( -2.78%)   rademacher  9774.3096 ( -2.41%)
k= 1024   gaussian 10102.6671 ( +0.87%)   rademacher  9777.1489 ( -2.38%)
k= 4096   gaussian 10095.2960 ( +0.80%)   rademacher  9826.9935 ( -1.88%)


Both converge to $\|\Delta\mathbf{H}\|_F^2$ — and look how **slowly**. A single probe
can be off by 50–80%; it takes $k \sim 10^2$ to get inside a couple of percent.

That is the variance formula doing exactly what it said it would. $\Delta\mathbf{H}$
here has its error concentrated in a few directions, and
$\operatorname{Var} \propto \sum_i s_i^4$ is brutal when the singular value spectrum is
top-heavy. **This is not an argument against the estimator** — training takes many
gradient steps, each with a fresh probe, so the *optimiser* does the averaging. It is
an argument against ever reading one probe as a measurement of anything.

Rademacher probes are often claimed to have lower variance for this family of
estimators. **This run does not establish that**: at $k = 16$ Rademacher is much
closer, at $k = 64$ Gaussian is, and the two are not separated at the sample sizes
above. Treat it as an open ablation, which is how this project has it recorded —
not as a known win.

### PHL's loss, as published

```python
forces   = -torch.autograd.grad(E.sum(), coords, create_graph=True)[0]
hvp      = -torch.autograd.grad(forces, coords, grad_outputs=v, create_graph=True)[0]
true_hvp = torch.bmm(true_hessian, v_flat).squeeze(-1)

energy_loss = (mse(E_pred, E_true) / sqrt(N)).mean()
force_loss  = (mse(F_pred, F_true).sum((1, 2)) / (3 * N)).mean()
hvp_loss    = (mse(hvp, true_hvp).sum((1, 2)) / (9 * N ** 2)).mean()

loss = energy_loss + 0.30 * force_loss + 0.09 * hvp_loss
```

with `v = torch.randn(3N)` redrawn every step, one probe per molecule per step.

**Now the part where we disagree with it.**

# 6. The "P" in PHL: why a raw probe is the wrong probe

The estimator above is unbiased for $\|\Delta\mathbf{H}\|_F^2$. But **the Frobenius
distance is not the quantity `openQHA` needs**, and the gap between the two is not
small. Two corrections, each with a derivation and each with a criterion that can
fail.

## 6.1 Correction 1 — project out the rigid-body subspace

Translational invariance gives an *exact identity*: $\mathbf{H}\,\mathbf{t}_\alpha = 0$
for the three translation vectors. Rotational invariance gives
$\mathbf{H}\,\mathbf{r}_\beta = -\partial(\nabla E)/\partial\theta_\beta$, which is zero
**only at a stationary point**.

Two consequences, and they are different from each other:

**At a minimum.** All six rigid directions are exactly zero in *both*
$\mathbf{H}_{\text{pred}}$ and $\mathbf{H}_{\text{ref}}$. An isotropic probe puts
$6/3N$ of its expected squared norm into those directions — for our molecules:

In [13]:
print("%-8s %6s %12s" % ("atoms", "3N", "wasted"))
for n in (8, 10, 15, 19):
    print("%-8d %6d %11.1f%%" % (n, 3 * n, 100 * 6 / (3 * n)))

atoms        3N       wasted
8            24        25.0%
10           30        20.0%
15           45        13.3%
19           57        10.5%


**10.5% to 25% of every probe is spent confirming an identity.** Not wrong — wasted.
And since the variance of the estimator scales with the fraction of probe mass doing
nothing useful, waste here costs convergence directly.

**Away from a minimum** — which is where the interesting training configurations live
— the rotational block is *not* zero. What it contains is the reference program's own
**residual gradient**. Fitting it teaches the model to reproduce the convergence
tolerance of whatever code generated the labels. That is worse than waste.

**The correction.** Build the mass-weighted Eckart projector onto the internal
subspace and push the probe through it:

$$
\mathbf{P} = \mathbf{I} - \sum_{\alpha}\mathbf{t}_\alpha\mathbf{t}_\alpha^\top
                        - \sum_{\beta}\mathbf{r}_\beta\mathbf{r}_\beta^\top ,
\qquad
\tilde{\mathbf{v}} = \sqrt{3N-6}\;\frac{\mathbf{P}\mathbf{v}}{\|\mathbf{P}\mathbf{v}\|}
$$

Then $\mathbb{E}[\tilde{\mathbf{v}}\tilde{\mathbf{v}}^\top] = \mathbf{P}$ and

$$
\mathbb{E}\!\left[\|\Delta\mathbf{H}\tilde{\mathbf{v}}\|^2\right]
= \operatorname{tr}(\Delta\mathbf{H}^\top\Delta\mathbf{H}\,\mathbf{P})
= \|\Delta\mathbf{H}\,\mathbf{P}\|_F^2
\;\;\longrightarrow\;\;
\|\mathbf{P}\,\Delta\mathbf{H}\,\mathbf{P}\|_F^2
$$

symmetrising. **That inner block is exactly what the frequencies depend on.**

## 6.2 Correction 2 — mass weighting

Frequencies come from $\mathbf{M}^{-1/2}\mathbf{H}\mathbf{M}^{-1/2}$. A Cartesian
Frobenius loss weights an H–H block and a C–C block equally, while for frequencies the
H block matters **12× more** ($1/\sqrt{m_\mathrm{H}m_\mathrm{H}}$ against
$1/\sqrt{m_\mathrm{C}m_\mathrm{C}}$).

The fix costs nothing, because

$$\big\|\mathbf{M}^{-1/2}\Delta\mathbf{H}\,\mathbf{M}^{-1/2}\,(\mathbf{M}^{1/2}\mathbf{w})\big\|
= \big\|\mathbf{M}^{-1/2}\Delta\mathbf{H}\,\mathbf{w}\big\|$$

> **Put the mass factor in the probe and the loss becomes mass-weighted for free.**
> You never build $\Delta\tilde{\mathbf{H}}$.

**The cost, stated rather than hidden:** a mass-weighted loss is the right one for
frequencies and thermodynamics, and the *wrong* one for forces under displacement.
`openQHA`'s product is free energy, so it takes mass weighting — and reports both.

In [14]:
def internal_probe(atoms, rng, mass_weighted=True):
    # A probe living in the internal subspace, optionally mass-weighted.
    # This is the production construction (see scripts/calibration/
    # s0_C_committee_calibration.py) -- not a tutorial simplification.
    m = atoms.get_masses()
    m3 = np.repeat(m, 3)
    V, _s, _rank = hessian.rigid_body_vectors(m, atoms.get_positions())
    P = np.eye(len(m3)) - V @ V.T          # V is already orthonormal
    v = P @ rng.standard_normal(len(m3))
    if mass_weighted:
        v = v / np.sqrt(m3)                # M^(-1/2) v
    return v / np.linalg.norm(v) * np.sqrt(len(m3) - V.shape[1])

Vr, _, _ = hessian.rigid_body_vectors(atoms.get_masses(), atoms.get_positions())

def rigid_fraction(v):
    return (np.linalg.norm(Vr.T @ v) ** 2) / (np.linalg.norm(v) ** 2)

# 6/3N is an EXPECTATION over probes, not a property of any one draw -- a single
# gaussian vector scatters widely around it. Average, and say how many.
n_draw = 2000
g = np.random.default_rng(7)
raw_fracs  = [rigid_fraction(g.standard_normal(3 * N)) for _ in range(n_draw)]
proj_fracs = [rigid_fraction(internal_probe(atoms, g, mass_weighted=False))
              for _ in range(n_draw)]

print("expected fraction in the rigid subspace, 6/3N = %.5f" % (6 / (3 * N)))
print("raw gaussian : mean %.5f   one draw can be %.5f to %.5f"
      % (np.mean(raw_fracs), min(raw_fracs), max(raw_fracs)))
print("projected    : mean %.5f   max over %d draws %.2e"
      % (np.mean(proj_fracs), n_draw, max(proj_fracs)))

expected fraction in the rigid subspace, 6/3N = 0.20000
raw gaussian : mean 0.20011   one draw can be 0.00996 to 0.66899
projected    : mean 0.00000   max over 2000 draws 2.24e-31


On average the raw probe puts exactly $6/3N$ of itself in the rigid subspace, as
derived — **but a single draw scatters widely around that**, which is worth seeing
before you trust any one-probe diagnostic. The projected probe puts nothing there, to
machine precision, every time.

## 6.3 Does the correction actually help? Measure it.

The theoretical variance reduction is at least $3N/(3N-6)$ — the fraction of the probe
that stops being wasted. **If the measured ratio does not clear that bound, the
implementation is wrong.** This is a criterion that can fail, which is the only kind
worth writing.

In [15]:
def hvp_loss_samples(dH, atoms, n, rng, projected, mass_weighted=False):
    out = []
    m3 = np.repeat(atoms.get_masses(), 3)
    V, _, _ = hessian.rigid_body_vectors(atoms.get_masses(), atoms.get_positions())
    P = np.eye(len(m3)) - V @ V.T
    for _ in range(n):
        v = rng.standard_normal(len(m3))
        if projected:
            v = P @ v
            if mass_weighted:
                v = v / np.sqrt(m3)
            v = v / np.linalg.norm(v) * np.sqrt(len(m3) - V.shape[1])
        out.append(np.linalg.norm(dH @ v) ** 2)
    return np.array(out)

n_s = 4000
raw  = hvp_loss_samples(dH, atoms, n_s, np.random.default_rng(11), projected=False)
proj = hvp_loss_samples(dH, atoms, n_s, np.random.default_rng(11), projected=True)

rel_raw  = raw.std()  / raw.mean()
rel_proj = proj.std() / proj.mean()
bound    = (3 * N) / (3 * N - 6)

print("relative std, raw probe        : %.4f" % rel_raw)
print("relative std, projected probe  : %.4f" % rel_proj)
print("variance ratio (raw/projected) : %.3f" % ((raw.var()) / (proj.var())))
print("theoretical lower bound 3N/(3N-6) = %.3f" % bound)
print()
print("PASS" if (raw.var() / proj.var()) >= bound else "FAIL -- check the implementation")

relative std, raw probe        : 0.6321
relative std, projected probe  : 0.5507
variance ratio (raw/projected) : 1.611
theoretical lower bound 3N/(3N-6) = 1.250

PASS


### A third correction, still undecided

There is one more, and it is the one with the largest effect on free energy — recorded
here as an open question rather than a solved problem, because it is one.

$\nu \propto \sqrt{\lambda}$, so $d\nu/d\lambda \propto 1/\nu$: a *fixed* absolute error
in a Hessian eigenvalue produces a much larger frequency error at the low end. And the
free-energy sensitivity runs the same way — measured in this project:

| frequency | $dG/d\nu$ (kcal·mol⁻¹·cm) |
|---|---|
| 3000 cm⁻¹ | 1.43 × 10⁻³ |
| **100 cm⁻¹** | **6.04 × 10⁻³** (**4.2×**) |

Both factors point the same direction, so **a plain Frobenius loss systematically
under-weights exactly the modes that dominate the free energy.** Three candidate fixes
(spectral shaping of the probe; an explicit low-frequency penalty; or change nothing
and merely *report* the low-frequency error) are on the table and not yet chosen. The
honest position is that this is a known bias with a known sign and no committed fix.

In [16]:
nu = np.linspace(50, 3200, 400)
dg = np.abs(thermo.da_dnu_kcal_per_cm(nu, 298.15))
import matplotlib.pyplot as plt
plt.figure(figsize=(6.2, 3.6))
plt.plot(nu, dg)
for f in (100, 500, 1500, 3000):
    plt.plot([f], [abs(thermo.da_dnu_kcal_per_cm(np.array([f]), 298.15)[0])], "o")
    plt.annotate("%d cm⁻¹" % f, (f, abs(thermo.da_dnu_kcal_per_cm(np.array([f]), 298.15)[0])),
                 textcoords="offset points", xytext=(6, 6), fontsize=8)
plt.xlabel("frequency / cm⁻¹"); plt.ylabel("|dA/dν| / kcal mol⁻¹ cm")
plt.title("Free-energy sensitivity is 4.2× larger at 100 than at 3000 cm⁻¹")
plt.tight_layout(); plt.show()

/tmp/ipykernel_23200/3869468674.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


# 7. Putting it together: a corrected PHL loss

Everything above assembles into a loss you could drop into a fine-tuning loop. The
structure is PHL's; the probe is ours.

In [17]:
def phl_loss_terms(atoms, calc, H_ref, rng, n_probes=1,
                   w_force=0.30, w_hvp=0.09, project=True, mass_weight=True):
    # One structure's contribution to a PHL loss.
    # Written with numpy and an explicit H_pred so the pieces are visible. In a
    # real training loop H_pred never exists: `hvp` comes from the double-backward
    # shown in section 4, which is what makes the curvature term affordable.
    n = len(atoms)
    m3 = np.repeat(atoms.get_masses(), 3)
    H_pred, _ = hessian.analytic_hessian(atoms, calc)

    V, _, _ = hessian.rigid_body_vectors(atoms.get_masses(), atoms.get_positions())
    P = np.eye(3 * n) - V @ V.T

    acc = 0.0
    for _ in range(n_probes):
        v = rng.standard_normal(3 * n)
        if project:
            v = P @ v
            if mass_weight:
                v = v / np.sqrt(m3)
            v = v / np.linalg.norm(v) * np.sqrt(3 * n - V.shape[1])
        acc += np.linalg.norm((H_pred - H_ref) @ v) ** 2 / (9 * n ** 2)
    return acc / n_probes


rng2 = np.random.default_rng(20260903)
for label, proj, mw in (("raw PHL          ", False, False),
                        ("+ projection     ", True,  False),
                        ("+ mass weighting ", True,  True)):
    vals = [phl_loss_terms(atoms, calc, H_ref, np.random.default_rng(s),
                           project=proj, mass_weight=mw) for s in range(40)]
    print("%s  mean %.6e   relative std %.3f"
          % (label, np.mean(vals), np.std(vals) / np.mean(vals)))

raw PHL            mean 1.189450e+01   relative std 0.651


+ projection       mean 1.011656e+01   relative std 0.540


+ mass weighting   mean 5.171896e+00   relative std 0.582


Three things to read off that table:

- The **mean changes** between rows. It should — each row is estimating a *different*
  quantity ($\|\Delta H\|_F^2$, then $\|\Delta H P\|_F^2$, then the mass-weighted
  version). Only the last one is the quantity frequencies depend on.
- The **relative spread falls with projection** (0.65 → 0.54), as derived.
- **Mass weighting does not reduce the spread further** — it moves the mean by roughly
  a factor of two, because hydrogen blocks now carry the weight they carry in the
  physics. It is a correction to *what is being estimated*, not to how noisily. Do not
  advertise it as a variance reduction; it is not one.

### The loss weights

`0.30` and `0.09` come from the PHL reference implementation. In this project they are
recorded as a **starting point, not a settled value** — they were fitted for a
different architecture, a different reference level, and a different target quantity,
and they must be swept.

Treat any weight you inherited from another paper as a prior, not a result.

# 8. Where the training structures come from — a trap worth naming

One last thing, because this project fell into it twice and caught itself twice.

**Do not evaluate on the model's own minima.** At a MACE minimum the MACE forces are
$\approx 0$ by construction (measured: $|F|_{max} = 5\text{–}7\times10^{-5}$ eV/Å), so
"error against the reference" degenerates into "the reference force itself". The tell
was that the reported deviation, 0.3790, was *identical* to the RMS of the reference
forces — the number was not measuring what it claimed.

The same trap in the curvature channel: a reference Hessian computed at a *MACE*
minimum gave **two imaginary frequencies** (−337.9, −288.2 cm⁻¹), because a minimum of
one surface is not a stationary point of the other.

So training and benchmark configurations come from three places, with the proportions
written into the record:

1. basins from the conformer search (MACE minima),
2. **thermal displacements** — quantum harmonic oscillator fluctuations, capped at
   0.15 Å (without the cap, acetone samples 0.542 Å RMS displacement, because normal
   modes are a linearisation and a large excursion along an 83 cm⁻¹ torsion wrecks
   bond lengths),
3. frames from real 298 K trajectories.

In [18]:
samples, th = hessian.thermal_displacements(atoms, calc, temperature_K=298.15,
                                            n_samples=3, seed=20260903)
print("samples drawn        :", len(samples))
print("modes sampled        :", th["n_modes_sampled"], "of", 3 * N - 6)
print("RMS displacement / Å :", np.round(th["rms_displacement_A"], 4))
print("ceiling / Å          :", th["max_rms_displacement_A"])
print("draws rejected       :", th["n_draws_rejected"], " <- the ceiling doing its job")
print("softest mode / cm⁻¹  : %.2f" % min(th["frequencies_cm_inv"]))

samples drawn        : 3
modes sampled        : 24 of 24
RMS displacement / Å : [0.0829 0.1163 0.1325]
ceiling / Å          : 0.15
draws rejected       : 0  <- the ceiling doing its job
softest mode / cm⁻¹  : 222.61


**Read that `draws rejected: 0` correctly.** It does *not* mean the ceiling is working;
it means the ceiling **did not bind** on this draw. All three samples came out at
0.08–0.13 Å, comfortably under 0.15 Å, so nothing was rejected and the guard was never
tested here.

The guard exists because of a different measurement: without a ceiling, acetone has
been observed to sample **0.542 Å** RMS displacement. A normal mode is a
*linearisation*, and a large excursion along a very soft torsion does not stay on the
torsion — it stretches bonds, and you are no longer sampling the molecule you meant to.

Note also that this particular structure's softest projected mode is ~223 cm⁻¹, which
is stiff as these things go. On a molecule with a genuinely floppy torsion the ceiling
binds often, and `n_draws_rejected` becomes a number you have to look at rather than a
zero you can skip past.

**One inconsistency worth naming.** `thermal_displacements` still builds its Hessian by
finite differences, while everything else in this notebook has moved to the analytic
one. For *sampling* that is defensible — the ±9–20 cm⁻¹ resolution shifts the
displacement amplitudes slightly and nothing downstream applies a threshold to them —
but it is an inconsistency, not a decision, and it is on the list to fix rather than
in the record as justified.

# 9. Tasks

**1 — Probe count.** Section 6.3 fixed the probe count at 1. Measure the relative
standard deviation of the PHL curvature term as a function of $k$ and confirm it falls
as $1/\sqrt{k}$. At what $k$ does it stop being worth the extra backward passes?

**2 — Rademacher.** Swap the Gaussian probe for $\pm1$ and repeat the variance
measurement, with and without projection. Does the lower-variance claim survive
projection, or does normalising $\mathbf{P}\mathbf{v}$ undo it?

**3 — Away from the minimum.** Displace the geometry by 0.1 Å and recompute
$\|\mathbf{P}\mathbf{H}\mathbf{P} - \mathbf{H}\|_F / \|\mathbf{H}\|_F$. At a stationary
point this should be $<10^{-6}$. Away from one it is not — and the number you get *is*
the amount of reference residual gradient the unprojected loss would have fitted.

**4 — The low-frequency bias.** Take $\Delta\mathbf{H}$ from section 5, diagonalise
both Hessians, and plot frequency error against frequency. Then weight each mode by
$dG/d\nu$ and re-plot. How much of the free-energy error lives below 300 cm⁻¹?

**5 — The hard one.** Write a spectrally-shaped probe
$\tilde{\mathbf{v}} = \mathbf{M}^{-1/2}\mathbf{P} f(\mathbf{H}_{\text{ref}})\mathbf{v}$
that up-weights low-frequency modes, and check on a molecule where you can afford both
full Hessians that the estimator still converges to the intended quantity. This is
correction 3 above — genuinely open, and the reason it is open is that nobody has
shown it does not break unbiasedness.

---

## What to take away

1. **A Hessian from a foundation model is one line and one extra backward pass**, and
   it is both more accurate and cheaper than finite differences. If you are still
   doing $6N$ force calls, stop.
2. **The asymmetry residual is a free accuracy diagnostic** — use it on every
   structure.
3. **Project before you diagonalise.** "Smallest six" is not "rigid six".
4. **HVPs make curvature supervision affordable**: $\mathcal{O}(1)$ instead of
   $\mathcal{O}(3N)$, and the Hutchinson identity makes it *unbiased*, not merely
   cheap.
5. **Unbiased for the wrong quantity is still wrong.** Raw PHL estimates the Cartesian
   Frobenius distance; frequencies depend on the mass-weighted internal block. Two
   corrections fix it, both of them free, and both of them measurable.
6. **A correction you have not measured is a guess.** The variance ratio in 6.3 has a
   theoretical lower bound; if your implementation misses it, your implementation is
   wrong, not the bound.